# CNN with Tanh Activation on CIFAR-10

This is a **separate file** as required by the lab assignment.

Same CNN architecture as the LeakyReLU experiment, but with **Tanh** activation and **Adam** optimizer (lr=0.0001).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

batch_size = 64
learning_rate = 0.0001
num_epochs = 20

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

## Download & Prepare CIFAR-10 Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(trainset)}")
print(f"Test samples: {len(testset)}")

## CNN Model with Tanh Activation

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Tanh(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Tanh(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Tanh(),
            nn.MaxPool2d(2, 2),
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.Tanh(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

model = SimpleCNN().to(device)
print(model)

## Training with Adam Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

writer = SummaryWriter('runs/cnn_tanh_adam')

train_losses, test_losses = [], []
train_accuracies, test_accuracies = [], []

for epoch in range(num_epochs):
    # Training Phase
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(trainloader)
    train_accuracy = 100 * train_correct / train_total

    # Test Phase
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_epoch_loss = test_loss / len(testloader)
    test_accuracy = 100 * test_correct / test_total

    # Log to TensorBoard
    writer.add_scalars('Loss', {'Train': epoch_loss, 'Test': test_epoch_loss}, epoch)
    writer.add_scalars('Accuracy', {'Train': train_accuracy, 'Test': test_accuracy}, epoch)

    train_losses.append(epoch_loss)
    test_losses.append(test_epoch_loss)
    train_accuracies.append(train_accuracy)
    test_accuracies.append(test_accuracy)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {train_accuracy:.2f}% | "
          f"Test Loss: {test_epoch_loss:.4f}, Test Acc: {test_accuracy:.2f}%")

writer.close()
print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")

## Per-Class Accuracy & Plots

In [ ]:
# Per-Class Accuracy
model.eval()
class_correct = [0] * 10
class_total = [0] * 10

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        for i in range(labels.size(0)):
            label = labels[i].item()
            class_correct[label] += (predicted[i] == label).item()
            class_total[label] += 1

class_acc = [100 * class_correct[i] / class_total[i] for i in range(10)]
print("--- Per-Class Accuracy (Tanh + Adam) ---")
for i in range(10):
    print(f"  {classes[i]:>10s}: {class_acc[i]:.2f}%")
print(f"\nOverall Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
epochs_range = range(1, num_epochs + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_range, train_losses, label='Train Loss')
axes[0].plot(epochs_range, test_losses, label='Test Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss (Tanh + Adam)')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(epochs_range, train_accuracies, label='Train Accuracy')
axes[1].plot(epochs_range, test_accuracies, label='Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy (Tanh + Adam)')
axes[1].legend()
axes[1].grid(True)

# Per-Class
axes[2].bar(classes, class_acc, color='steelblue')
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Per-Class Accuracy (Tanh + Adam)')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y')

plt.tight_layout()
plt.show()